In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import joblib
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from lime.lime_tabular import LimeTabularExplainer
from datetime import datetime
from scripts.a_data_merger import perform_merge
from scripts.a_preprocessor import bug_checker, normalization_data
from scripts.a_results import explainability
from config import NAME_FILE_BUGS, NAME_FILE_METRICS_CODE, UMBRAL

# to do metrics extraction before
mode = 'predict'

#--------------------------------------LOADING SOURCES ------------------------------------------
# load model
models_folder = os.path.join('..', 'models')
os.makedirs(models_folder, exist_ok=True)
bug_predictor_path = os.path.join(models_folder, 'bug_predictor.pkl')
kmeans_path = os.path.join(models_folder, "kmeans.pkl")
brf_model, feature_names = joblib.load(bug_predictor_path)
kmeans = joblib.load(kmeans_path)
ruta_archivo = os.path.join(models_folder, "x_train_for_lime.csv")
X_train = pd.read_csv(ruta_archivo)

# path
data_folder = os.path.join("..", "data", "extractions")
if not os.path.exists(data_folder):
    os.makedirs(data_folder)

info_folder = os.path.join("..", "data", "info")
if not os.path.exists(info_folder):
    os.makedirs(info_folder)

predictions_folder = os.path.join("..", "data", "predictions")
if not os.path.exists(predictions_folder):
    os.makedirs(predictions_folder)

os.makedirs("results", exist_ok=True)

# load code metrics
code_data_path = os.path.join(data_folder, NAME_FILE_METRICS_CODE)
if os.path.exists(code_data_path):
    metrics_data = pd.read_csv(code_data_path)
else:
    raise FileNotFoundError(f"File not found {code_data_path}")

# load data from bugs
bugs_data_path = os.path.join(data_folder, NAME_FILE_BUGS)
if os.path.exists(bugs_data_path):
    bugs_data = pd.read_csv(bugs_data_path, sep=';')
else:
    raise FileNotFoundError(f"File not found {bugs_data_path}")

#---------- load dataset for prediction--------------------------

predict_data_path = os.path.join(predictions_folder, 'commit_set_prediction.csv')
if os.path.exists(predict_data_path):
    predict_data = pd.read_csv(predict_data_path, sep='|')
    
else:
    raise FileNotFoundError(f"File not found {predict_data_path}")


# ---------------------------------------PREPROCESSING --------------------------------------------

# merge
merged_data = perform_merge(predict_data, bugs_data, metrics_data, mode)

merged_data['Bug'] = 0

# bug validation
merged_data = bug_checker(merged_data)

# normalization of data
X_scaled = normalization_data(merged_data, mode)

#define y
y = X_scaled['Bug'] 
X_scaled = X_scaled.drop(columns=['Bug'])
X_scaled['cluster'] = kmeans.predict(X_scaled)
print("Records of final data set:", X_scaled.shape[0])

#---------------------------------------PREDICTION ------------------------------------------------------------------------------------    

print("Columns in final_dataset BEFORE order:")
print(list(X_scaled.columns))

print("Columns expected by model (feature_names):")
print(feature_names)

final_dataset = X_scaled[feature_names]

print("Columns in final_dataset AFTER order:")
print(list(final_dataset.columns))


probas = brf_model.predict_proba(final_dataset)[:, 1]

# get binary predictions (0 - 1) using umbral
y_pred_bug = (probas >= UMBRAL).astype(int)

# how many files have been predicted 'with bug'
quantity_with_bug = np.sum(y_pred_bug)

# print the quantity
print(f"Quantity of files with positive prediction for Bug: {quantity_with_bug} from a total of {len(final_dataset)}")

# Create a new DataFrame only with columns that you want in the results
resultados = final_dataset.copy()

# Add columns of results
resultados['probabilidad_bug'] = probas
resultados['prediccion_bug'] = (probas >= UMBRAL).astype(int)


# Load file_hashes_df to get names of files and hash
file_hashes_df = pd.read_csv(os.path.join(info_folder, 'file_hashes_backup.csv'))
file_hashes_df = file_hashes_df.drop_duplicates(subset=['file_name_unique'])
file_hashes_df = file_hashes_df.reset_index(drop=True)

# Filter only rows with prediction bug = 1
resultados_con_bug = resultados[resultados['prediccion_bug'] == 1].copy()

# join with file_hashes_df to get the real name of file
resultados_con_bug = resultados_con_bug.merge(
    file_hashes_df[['file_hash', 'file_name_unique']],
    how='left',
    on='file_hash'
)

# keep the columns 'file_hash' and 'file_name_unique', and remove duplicates if there are
resultados_finales = resultados_con_bug[['file_hash', 'file_name_unique']].drop_duplicates()

# save result in CSV with current date
fecha_actual = datetime.now().strftime('%Y-%m-%d')
nombre_archivo = f'prediction_results_{fecha_actual}.csv'
ruta_archivo = os.path.join("results", nombre_archivo)
resultados_finales.to_csv(ruta_archivo, index=False)

print(f"File saved with {len(resultados_finales)} predicted files with Bug in {nombre_archivo}")


# -------------------------------------------EXPLAINABILITY ------------------------------------------------------------------------------------

# Create a DataFrame with the probabilities, index and values of file_hash
probabilities_df = pd.DataFrame({    
    'file_hash': resultados['file_hash'].values,
    'probability_class_1': resultados['probabilidad_bug'].values,
    'prediction': resultados['prediccion_bug'].values
})

# Order by probabilities of class 1 descendent
probabilities_df = probabilities_df.sort_values(by='probability_class_1', ascending=False)
probabilities_df_unique = probabilities_df.drop_duplicates(subset=['file_hash'])

file_hashes_df = pd.read_csv(os.path.join(info_folder, 'file_hashes_backup.csv'))
file_hashes_df = file_hashes_df.drop_duplicates(subset=['file_name_unique'])
file_hashes_df = file_hashes_df.reset_index(drop=True)

probabilities_df_unique = probabilities_df_unique.merge(
    file_hashes_df[['file_hash', 'file_name_unique']],
    how='left',
    on='file_hash'
)
probabilities_path = os.path.join(predictions_folder, 'files_defects_predicted.csv')
probabilities_df_unique.to_csv(probabilities_path, index=False)
print("\nSaved in files_defects_predicted.csv")

print("Quantity of unique rows by file_hash:", file_hashes_df.shape[0])
print("¿Are there duplicates?:", file_hashes_df['file_hash'].duplicated().any())

# Print files with prediction = 1 in log
files_with_bugs = probabilities_df_unique[probabilities_df_unique['prediction'] == 1]
files_with_bugs_count = len(files_with_bugs)

print(f"\n=== FILES PREDICTED WITH BUGS (prediction = 1) ===")
print(f"Total files with bug prediction: {files_with_bugs_count}")

if files_with_bugs_count > 0:
    print("\nFiles predicted with bugs:")
    print("-" * 80)
    for idx, row in files_with_bugs.iterrows():
        file_name = row['file_name_unique'] if pd.notna(row['file_name_unique']) else f"Unknown (hash: {row['file_hash']})"
        probability = row['probability_class_1']
        print(f"{idx+1:3d}. {file_name:50s} | Probability: {probability:.4f}")
    print("-" * 80)
else:
    print("✓ No files were predicted with bugs (all predictions = 0)")

print("Quantity of unique rows by file_hash:", file_hashes_df.shape[0])
print("¿Are there duplicates?:", file_hashes_df['file_hash'].duplicated().any())


# Create explainer LIME with all features
explainer_lime = LimeTabularExplainer(
    training_data=np.array(X_train),  # all features (X_train)
    mode="classification",
    feature_names=X_train.columns,
    class_names=["Free Bug", "With Bug"],
    discretize_continuous=True
)
origin_instance = final_dataset

explainability(probabilities_df_unique, file_hashes_df, mode, brf_model, X_train, origin_instance)




Rows after first merge (commit_data + issue_data): 104
File saved with the first merge done for prediction in ..\data\preprocess\merge1_predict.csv
Rows after expanding lines: 4629
Data splitted saved in 02_commit_splitting_for predict.csv
File saved with the second merge done in ..\data\preprocess\03_merged_data.csv
Merged data columns: ['id', 'commit_id', 'comment', 'author', 'date', 'description', 'pull_request_number', 'tool_ticket_commit', 'num_files_added', 'num_files_deleted', 'num_files_edited', 'total_files', 'modified_files', 'jira_ticket', 'resume', 'creation', 'update', 'Rapporteur', 'Composants', 'Affecte la/les version(s)', 'version corrected', 'environment', 'squad', 'resolution', 'severity', 'file_name_commit', 'file_name', 'nloc', 'cyclomatic_complexity', 'density', 'complexity_rank', 'token_count', 'parameter_count', 'length', 'function_count', 'location']
Total rows with bugs: 207 out of 4629 (4.47%)
Dataframe with contributoirs and categoric columns saved in ..\data

c:\Users\B552LX\OneDrive - AXA\Documentos\SDP\software-defect-prediction\.venv12\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Explainability LIME saved in HTML explainability\lime_explanation_file_hash_993533222934.html
Explainability LIME saved in text explainability\lime_explanation_file_hash_993533222934.txt

Generating explainability LIME for file_hash: 983933933048 (File: RenouvellementEntity.java, Probability bug: 0.93)


c:\Users\B552LX\OneDrive - AXA\Documentos\SDP\software-defect-prediction\.venv12\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Explainability LIME saved in HTML explainability\lime_explanation_file_hash_983933933048.html
Explainability LIME saved in text explainability\lime_explanation_file_hash_983933933048.txt

Generating explainability LIME for file_hash: 400535714442 (File: DonneeHistoriseeMapper.java, Probability bug: 0.92)


...

Explainability LIME saved in HTML explainability\lime_explanation_file_hash_63088205630.html
Explainability LIME saved in text explainability\lime_explanation_file_hash_63088205630.txt

Generating explainability LIME for file_hash: 61435840986 (File: LienAssistanceConventionsEntity.java, Probability bug: 0.85)


c:\Users\B552LX\OneDrive - AXA\Documentos\SDP\software-defect-prediction\.venv12\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Explainability LIME saved in HTML explainability\lime_explanation_file_hash_61435840986.html
Explainability LIME saved in text explainability\lime_explanation_file_hash_61435840986.txt

Generating explainability LIME for file_hash: 8418683966 (File: GarantieSouscriteEntity.java, Probability bug: 0.85)


c:\Users\B552LX\OneDrive - AXA\Documentos\SDP\software-defect-prediction\.venv12\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Explainability LIME saved in HTML explainability\lime_explanation_file_hash_8418683966.html
Explainability LIME saved in text explainability\lime_explanation_file_hash_8418683966.txt
10 
 explanaitions generated
